# Classical ML Classifiers for Token-Level NER

In this notebook, we train and compare several classical machine learning classifiers for a small token-level NER-style task.

The goal is not to build a perfect NER system. The goal is to understand how classical ML models use handcrafted features to predict entity labels.

## Learning Objectives

By the end of this notebook, you should be able to:
- transform token-level feature dictionaries into numerical vectors
- train classical ML classifiers for token labeling
- compare different classifiers
- inspect model errors
- understand why feature engineering matters

---

## 1. Setup

We use `spaCy` for tokenization and POS tagging, and `scikit-learn` for classical machine learning models.

In [1]:
import spacy
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split


In [2]:
nlp = spacy.load("en_core_web_sm")

## 2. Tiny NER Dataset

We create a small manually labeled dataset.

Each example consists of:

- a sentence
- one label per token

We use BIO-style labels:

- `B-ORG`: beginning of an organization
- `B-PER`: beginning of a person
- `B-GPE`: beginning of a geopolitical entity/location
- `B-DATE`: beginning of a date
- `O`: outside any entity


In [3]:
dataset = [
    ("Google acquired DeepMind in London in 2014.",
     ["B-ORG", "O", "B-ORG", "O", "B-GPE", "O", "B-DATE", "O"]),

    ("Microsoft opened an office in Berlin.",
     ["B-ORG", "O", "O", "O", "O", "B-GPE", "O"]),

    ("Alice joined Amazon in Paris.",
     ["B-PER", "O", "B-ORG", "O", "B-GPE", "O"]),

    ("Bob works at Apple in California.",
     ["B-PER", "O", "O", "B-ORG", "O", "B-GPE", "O"]),

    ("OpenAI released ChatGPT in 2022.",
     ["B-ORG", "O", "B-ORG", "O", "B-DATE", "O"]),

    ("Sarah visited Hamburg last Monday.",
     ["B-PER", "O", "B-GPE", "O", "B-DATE", "O"]),

    ("Meta hired John in Dublin.",
     ["B-ORG", "O", "B-PER", "O", "B-GPE", "O"]),

    ("Tesla expanded operations in Austin.",
     ["B-ORG", "O", "O", "O", "B-GPE", "O"]),

    ("Anna moved to Munich in 2020.",
     ["B-PER", "O", "O", "B-GPE", "O", "B-DATE", "O"]),

    ("IBM acquired RedHat in 2019.",
     ["B-ORG", "O", "B-ORG", "O", "B-DATE", "O"]),
]

## 3. Check Tokenization

Before training anything, we check whether the number of tokens matches the number of labels.

This is important because NER is a token-level prediction task.

In [4]:
for sentence, labels in dataset:
    doc = nlp(sentence)
    tokens = [token.text for token in doc]
    print(sentence)
    print(tokens)
    print(labels)
    print("tokens:", len(tokens), "labels:", len(labels))
    print("OK:", len(tokens) == len(labels))
    print("-" * 80)


Google acquired DeepMind in London in 2014.
['Google', 'acquired', 'DeepMind', 'in', 'London', 'in', '2014', '.']
['B-ORG', 'O', 'B-ORG', 'O', 'B-GPE', 'O', 'B-DATE', 'O']
tokens: 8 labels: 8
OK: True
--------------------------------------------------------------------------------
Microsoft opened an office in Berlin.
['Microsoft', 'opened', 'an', 'office', 'in', 'Berlin', '.']
['B-ORG', 'O', 'O', 'O', 'O', 'B-GPE', 'O']
tokens: 7 labels: 7
OK: True
--------------------------------------------------------------------------------
Alice joined Amazon in Paris.
['Alice', 'joined', 'Amazon', 'in', 'Paris', '.']
['B-PER', 'O', 'B-ORG', 'O', 'B-GPE', 'O']
tokens: 6 labels: 6
OK: True
--------------------------------------------------------------------------------
Bob works at Apple in California.
['Bob', 'works', 'at', 'Apple', 'in', 'California', '.']
['B-PER', 'O', 'O', 'B-ORG', 'O', 'B-GPE', 'O']
tokens: 7 labels: 7
OK: True
----------------------------------------------------------------

## 4. Token-Level Feature Function

We now define a feature function.

For each token, we create handcrafted features such as:

- lowercase form
- capitalization
- prefixes and suffixes
- POS tag
- previous token
- next token
- gazetteer membership


In [5]:
ORG_GAZETTEER = {"google", "deepmind", "microsoft", "amazon", "apple", "openai", "chatgpt", "meta", "tesla", "ibm", "redhat"}
PER_GAZETTEER = {"alice", "bob", "sarah", "john", "anna"}
GPE_GAZETTEER = {"london", "berlin", "paris", "california", "hamburg", "dublin", "austin", "munich"}


In [6]:
def token_features(doc, i):
    token = doc[i]
    text = token.text
    lower = text.lower()

    features = {
        "bias": 1.0,
        "token.lower": lower,
        "token.length": len(text),
        "prefix_2": text[:2],
        "prefix_3": text[:3],
        "suffix_2": text[-2:],
        "suffix_3": text[-3:],
        "is_title": text.istitle(),
        "is_upper": text.isupper(),
        "is_digit": text.isdigit(),
        "is_alpha": token.is_alpha,
        "pos": token.pos_,
        "in_org_gazetteer": lower in ORG_GAZETTEER,
        "in_per_gazetteer": lower in PER_GAZETTEER,
        "in_gpe_gazetteer": lower in GPE_GAZETTEER,
    }

    if i > 0:
        prev = doc[i - 1]
        features.update({
            "prev.lower": prev.text.lower(),
            "prev.is_title": prev.text.istitle(),
            "prev.pos": prev.pos_,
        })
    else:
        features["BOS"] = True

    if i < len(doc) - 1:
        nxt = doc[i + 1]
        features.update({
            "next.lower": nxt.text.lower(),
            "next.is_title": nxt.text.istitle(),
            "next.pos": nxt.pos_,
        })
    else:
        features["EOS"] = True

    return features

## 5. Build Token-Level Training Data

We convert the sentence-level dataset into token-level examples.

Each token becomes one training example.

In [7]:
X_dicts = []
y = []
tokens_debug = []

for sentence, labels in dataset:
    doc = nlp(sentence)
    assert len(doc) == len(labels), f"Token/label mismatch in: {sentence}"

    for i, token in enumerate(doc):
        X_dicts.append(token_features(doc, i))
        y.append(labels[i])
        tokens_debug.append(token.text)

print("Number of token examples:", len(X_dicts))
print("Example token:", tokens_debug[0])
print("Example features:")
X_dicts[0]

Number of token examples: 65
Example token: Google
Example features:


{'bias': 1.0,
 'token.lower': 'google',
 'token.length': 6,
 'prefix_2': 'Go',
 'prefix_3': 'Goo',
 'suffix_2': 'le',
 'suffix_3': 'gle',
 'is_title': True,
 'is_upper': False,
 'is_digit': False,
 'is_alpha': True,
 'pos': 'PROPN',
 'in_org_gazetteer': True,
 'in_per_gazetteer': False,
 'in_gpe_gazetteer': False,
 'BOS': True,
 'next.lower': 'acquired',
 'next.is_title': False,
 'next.pos': 'VERB'}

## 6. Convert Feature Dictionaries into Vectors

scikit-learn models cannot directly process dictionaries.

We use `DictVectorizer` to transform feature dictionaries into numerical vectors.

In [8]:
vectorizer = DictVectorizer(sparse=True)
X = vectorizer.fit_transform(X_dicts)

print("Feature matrix shape:", X.shape)
print("Number of labels:", len(y))
print("Number of generated features:", len(vectorizer.get_feature_names_out()))

Feature matrix shape: (65, 322)
Number of labels: 65
Number of generated features: 322


## 7. Train/Test Split

We split the token examples into training and test data.

Note: This is a simplified setup. In real NER, splitting should usually happen at the sentence/document level to avoid leakage.

In [9]:
X_train, X_test, y_train, y_test, tok_train, tok_test = train_test_split(
    X, y, tokens_debug, test_size=0.30, random_state=42, stratify=y
)

print("Train examples:", X_train.shape[0])
print("Test examples:", X_test.shape[0])

Train examples: 45
Test examples: 20


## 8. Train Several Classical Classifiers

We compare different classical ML models:

- Logistic Regression
- Linear SVM
- Decision Tree
- Random Forest
- Naive Bayes

The goal is to compare behavior, not to maximize performance.

In [10]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Linear SVM": LinearSVC(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Naive Bayes": MultinomialNB(),
}

In [11]:
results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append({"model": name, "accuracy": acc})
    trained_models[name] = model

pd.DataFrame(results).sort_values("accuracy", ascending=False)

,model,accuracy
1,Linear SVM,0.95
2,Decision Tree,0.95
3,Random Forest,0.95
0,Logistic Regression,0.85
4,Naive Bayes,0.75


## 9. Inspect Classification Reports

Accuracy alone is not enough.

NER labels are often imbalanced because most tokens are `O`.

So we also inspect precision, recall, and F1-score.

In [12]:
for name, model in trained_models.items():
    print("=" * 80)
    print(name)
    print("=" * 80)
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, zero_division=0))

Logistic Regression
              precision    recall  f1-score   support

      B-DATE       1.00      1.00      1.00         2
       B-GPE       1.00      0.50      0.67         2
       B-ORG       0.60      1.00      0.75         3
       B-PER       1.00      0.50      0.67         2
           O       0.91      0.91      0.91        11

    accuracy                           0.85        20
   macro avg       0.90      0.78      0.80        20
weighted avg       0.89      0.85      0.85        20

Linear SVM
              precision    recall  f1-score   support

      B-DATE       1.00      1.00      1.00         2
       B-GPE       1.00      1.00      1.00         2
       B-ORG       0.75      1.00      0.86         3
       B-PER       1.00      1.00      1.00         2
           O       1.00      0.91      0.95        11

    accuracy                           0.95        20
   macro avg       0.95      0.98      0.96        20
weighted avg       0.96      0.95      0.95   

## 10. Error Analysis

Now we inspect which tokens were misclassified.

This is often more useful than just looking at a score.

In [13]:
best_model_name = pd.DataFrame(results).sort_values("accuracy", ascending=False).iloc[0]["model"]
best_model = trained_models[best_model_name]

y_pred = best_model.predict(X_test)

errors = []
for token, gold, pred in zip(tok_test, y_test, y_pred):
    if gold != pred:
        errors.append({"token": token, "gold": gold, "predicted": pred})

print("Best model:", best_model_name)
pd.DataFrame(errors)

Best model: Linear SVM


,token,gold,predicted
0,operations,O,B-ORG


## 11. Predict Labels for a New Sentence

Now we apply the trained models to a new sentence.

This helps us see whether the classifiers generalize beyond the tiny training data.

In [14]:
def predict_sentence(sentence, model):
    doc = nlp(sentence)
    features = [token_features(doc, i) for i in range(len(doc))]
    X_new = vectorizer.transform(features)
    preds = model.predict(X_new)

    return pd.DataFrame({
        "token": [token.text for token in doc],
        "predicted_label": preds
    })

In [15]:
new_sentence = "Amazon opened an office in Paris in 2023."

for name, model in trained_models.items():
    print("=" * 80)
    print(name)
    display(predict_sentence(new_sentence, model))

Logistic Regression


,token,predicted_label
0,Amazon,B-ORG
1,opened,O
2,an,O
3,office,O
4,in,O
5,Paris,B-GPE
6,in,O
7,2023,B-DATE
8,.,O


Linear SVM


,token,predicted_label
0,Amazon,B-ORG
1,opened,O
2,an,O
3,office,O
4,in,O
5,Paris,B-GPE
6,in,O
7,2023,B-DATE
8,.,O


Decision Tree


,token,predicted_label
0,Amazon,B-ORG
1,opened,O
2,an,O
3,office,O
4,in,O
5,Paris,B-GPE
6,in,O
7,2023,B-DATE
8,.,O


Random Forest


,token,predicted_label
0,Amazon,B-ORG
1,opened,O
2,an,O
3,office,O
4,in,O
5,Paris,B-GPE
6,in,O
7,2023,B-DATE
8,.,O


Naive Bayes


,token,predicted_label
0,Amazon,B-ORG
1,opened,O
2,an,O
3,office,O
4,in,O
5,Paris,B-GPE
6,in,O
7,2023,B-GPE
8,.,O


## 12. Inspect Logistic Regression Feature Weights

One advantage of classical linear models is that we can inspect learned feature weights.

This helps us understand what the model learned.

In [16]:
logreg = trained_models["Logistic Regression"]
feature_names = vectorizer.get_feature_names_out()

for class_idx, label in enumerate(logreg.classes_):
    coefs = logreg.coef_[class_idx]
    top_idx = coefs.argsort()[-10:][::-1]

    print("=" * 80)
    print("Top features for label:", label)
    print("=" * 80)
    for i in top_idx:
        print(f"{feature_names[i]:40s} {coefs[i]:.3f}")

Top features for label: B-DATE
next.pos=PUNCT                           0.419
next.lower=.                             0.419
is_digit                                 0.331
prefix_2=20                              0.331
pos=NUM                                  0.331
token.lower=monday                       0.271
prefix_2=Mo                              0.271
prefix_3=Mon                             0.271
prev.lower=last                          0.271
prev.pos=ADJ                             0.271
Top features for label: B-GPE
in_gpe_gazetteer                         0.637
prev.lower=in                            0.534
prev.pos=ADP                             0.534
is_title                                 0.341
pos=PROPN                                0.284
token.length                             0.282
next.pos=PUNCT                           0.280
next.lower=.                             0.280
suffix_2=in                              0.268
suffix_3=lin                             0.184

## 13. Experiment

Try changing the feature function.

For example:

- remove gazetteer features
- remove context features
- remove POS features
- add longer prefixes/suffixes

Then rerun the notebook and compare the classifier results.

### Question

Which feature group seems most useful for this task?

## 14. Reflection

Answer briefly:

1. Which classifier performed best?
2. Did all classifiers make the same mistakes?
3. Which labels were easiest to predict?
4. Which labels were hardest to predict?
5. Why is the `O` label often easy to predict?
6. Why is token-level classification not enough for full NER?
7. What role did feature engineering play in the results?

## Summary

In this notebook, we trained several classical ML classifiers for a token-level NER-style task.

We used:

- handcrafted token features
- gazetteer features
- contextual features
- `DictVectorizer`
- Logistic Regression
- Linear SVM
- Decision Tree
- Random Forest
- Naive Bayes

Main takeaway:

> Classical ML models can learn useful label patterns, but their performance depends strongly on the quality of handcrafted features.

Next, we will discuss why sequence-aware models such as CRFs became important for NER.